In [ ]:
from __future__ import annotations

import os
import sys
import importlib
from pathlib import Path

import requests

# =============================================================================
# CONFIG — edit these
# =============================================================================
REPO = Path("/home/jovyan/data-store/spectralbridge")

# Paste your NEON API token here (or set it in the environment beforehand)
os.environ["NEON_TOKEN"] = "Paste-your-token-here"

BASE_FOLDER = Path("YELL_t02")  # created under REPO if relative
SITE_CODE = "YELL"
YEAR_MONTH = "2023-07"             # YYYY-MM
PRODUCT_CODE = "DP1.30006.001"

FLIGHT_LINES = [
    "NEON_D12_YELL_DP1_L001-1_20230715_directional_reflectance",
    "NEON_D12_YELL_DP1_L001-1_20230718_directional_reflectance"
]

# ONLY these 2 combinations are allowed:
#   EXTRACTION_MODE="full",    POLYGON_PATH=None
#   EXTRACTION_MODE="polygon", POLYGON_PATH=<existing polygon file>
EXTRACTION_MODE = "polygon"
POLYGON_PATH = REPO / "merged_all_AOP_polygon_data_2023_2024.geojson"
# Example full-scene mode:
# EXTRACTION_MODE = "full"
# POLYGON_PATH = None

TOPO_FIT_MODE = "scene"  # "scene" or "tile"
ENGINE = "thread"
MAX_WORKERS = 1
# =============================================================================

# --- 1) Resolve repo / real package (avoid importing the repo folder itself) ---
if not (REPO / "src" / "spectralbridge" / "pipelines" / "pipeline.py").exists():
    raise FileNotFoundError(f"Bad REPO path / layout: {REPO}")

SRC = REPO / "src"

# Drop paths that make `import spectralbridge` bind to the repo directory
bad_parents = {REPO.parent.resolve(), REPO.resolve()}
cleaned = []
for p in sys.path:
    try:
        pr = Path(p).resolve() if p not in ("", ".") else Path.cwd().resolve()
    except Exception:
        cleaned.append(p)
        continue
    if pr in bad_parents:
        continue
    cleaned.append(p)
sys.path = cleaned
sys.path.insert(0, str(SRC))

%cd {REPO}
%pip install -e .

# Clear any previously imported wrong package
for k in list(sys.modules):
    if k == "spectralbridge" or k.startswith("spectralbridge."):
        del sys.modules[k]
importlib.invalidate_caches()

# --- 2) NEON token ---
NEON_TOKEN = os.environ.get("NEON_TOKEN", "").strip()
if not NEON_TOKEN or NEON_TOKEN == "Paste-your-token-here":
    raise ValueError("Set a real NEON token in os.environ['NEON_TOKEN']")

# --- 3) Import + patch download to send X-API-Token ---
import spectralbridge.envi_download as envi_download
import spectralbridge.pipelines.pipeline as pipeline
from spectralbridge import go_forth_and_multiply
import spectralbridge

print("Loaded spectralbridge from:", spectralbridge.__file__)
assert "/src/spectralbridge/" in str(spectralbridge.__file__)

_orig_download = envi_download.download_neon_file

def download_with_token(*args, **kwargs):
    session = requests.Session()
    session.headers["X-API-Token"] = NEON_TOKEN
    kwargs["session"] = session  # set/overwrite; don't pass twice
    return _orig_download(*args, **kwargs)

envi_download.download_neon_file = download_with_token
pipeline.download_neon_file = download_with_token

# --- 4) Validate extraction mode ---
extraction_mode = str(EXTRACTION_MODE).strip().lower()
if extraction_mode not in ("full", "polygon"):
    raise ValueError(f"EXTRACTION_MODE must be 'full' or 'polygon', got {EXTRACTION_MODE!r}")

if extraction_mode == "full":
    if POLYGON_PATH is not None:
        raise ValueError("EXTRACTION_MODE='full' requires POLYGON_PATH=None")
    polygon_path = None
else:
    if POLYGON_PATH is None:
        raise ValueError("EXTRACTION_MODE='polygon' requires POLYGON_PATH=<file>")
    polygon_path = Path(POLYGON_PATH).expanduser().resolve()
    if not polygon_path.exists():
        raise FileNotFoundError(f"Polygon file not found: {polygon_path}")

base_folder = BASE_FOLDER.expanduser()
base_folder = base_folder.resolve() if base_folder.is_absolute() else (REPO / base_folder).resolve()
base_folder.mkdir(parents=True, exist_ok=True)

# --- 5) Run ---
print("=" * 80)
print("Full NEON pipeline (download + token)")
print(f"  repo={REPO}")
print(f"  base_folder={base_folder}")
print(f"  site_code={SITE_CODE}  year_month={YEAR_MONTH}")
print(f"  flight_lines={FLIGHT_LINES}")
print(f"  extraction_mode={extraction_mode}  polygon_path={polygon_path}")
print("=" * 80)

kwargs = {
    "base_folder": base_folder,
    "site_code": SITE_CODE,
    "year_month": YEAR_MONTH,
    "product_code": PRODUCT_CODE,
    "flight_lines": list(FLIGHT_LINES),
    "engine": ENGINE,
    "max_workers": MAX_WORKERS,
    "extraction_mode": extraction_mode,
    "polygon_path": polygon_path,
    "topo_fit_mode": TOPO_FIT_MODE,
}
if extraction_mode == "polygon":
    kwargs.update(
        {
            "polygon_overwrite": False,
            "polygon_min_overlap": 0.0,
            "polygon_search_buffer_m": 0.0,
        }
    )

go_forth_and_multiply(**kwargs)

print("\nDone")
for _fl in FLIGHT_LINES:
    print(f"  H5 → {base_folder / (_fl + '.h5')}")
    print(f"  work dir → {base_folder / _fl}")

In [ ]:
#GO COMMAND INIT
import pexpect

child = pexpect.spawn("./gocmd init", encoding="utf-8")

# 1. Defaults
child.expect("Host")
child.sendline("")

child.expect("Port")
child.sendline("")

child.expect("Zone")
child.sendline("")

# 2. Username (ask user in notebook)
child.expect("Username")
username = input("Enter Cyverse Account Username: ")
child.sendline(username)

# 3. Password (secure input)
child.expect("Password")

from getpass import getpass
password = getpass("Enter Cyverse Account Password: ")
child.sendline(password)

# 4. Finish
child.expect(pexpect.EOF)

print("\n✅ gocmd init completed")

In [ ]:
%cd /home/jovyan/data-store/spectralbridge/

import importlib
import sys
from pathlib import Path

repo_root = Path("/home/jovyan/data-store/spectralbridge")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import move_folders_from_instance_to_remote as uploader
importlib.reload(uploader)

uploader.run_transfer(
    "/home/jovyan/data-store/spectralbridge/YELL_t02",
    "i:/iplant/home/shared/earthlab/macrosystems/Aug_2026_Processed_Flightlines",
)

In [ ]:
%cd /home/jovyan/data-store/spectralbridge/

print("Removing transferred local folder: /home/jovyan/data-store/spectralbridge/YELL_t02")
!rm -rf /home/jovyan/data-store/spectralbridge/YELL_t02
print("Done")